In [2]:
import pandas as pd
import gradio as gr
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import shap

# DATA COLLECTION

file_path = "/content/scholars_balanced_200k.csv"

try:
    df = pd.read_csv(file_path)

    df.columns = [
        c.strip().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]

    print("Dataset loaded successfully.")
    print("Dataset shape:", df.shape)

except FileNotFoundError:
    print("ERROR: scholars_balanced_200k.csv not found.")
    raise

# CHECK REQUIRED COLUMNS

FEATURES = [
    "Education_Qualification",
    "Gender",
    "Community",
    "Religion",
    "Exservice_men",
    "Disability",
    "Sports",
    "Annual_Percentage",
    "Income",
    "India"
]

required_columns = FEATURES + ["Outcome"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in dataset: {missing_columns}"
    )

# ENCODING MAPS

education_map = {
    "Undergraduate": 0,
    "Postgraduate": 1,
    "Doctorate": 2
}

gender_map = {
    "Male": 0,
    "Female": 1
}

community_map = {
    "General": 0,
    "OBC": 1,
    "Minority": 2,
    "SC/ST": 3
}

religion_map = {
    "Hindu": 0,
    "Muslim": 1,
    "Christian": 2,
    "Others": 3
}

yes_no_map = {
    "No": 0,
    "Yes": 1
}

percentage_map = {
    "Below 50": 0,
    "50-60": 1,
    "60-70": 2,
    "70-80": 3,
    "80-90": 4,
    "Above 90": 5
}

income_map = {
    "Upto 1.5L": 0,
    "1.5L to 3L": 1,
    "3L to 6L": 2,
    "Above 6L": 3
}

india_map = {
    "In": 0,
    "Out": 1
}

# REMOVE MISSING TARGET VALUES

print("\nMissing Outcome values before cleaning:")
print(df["Outcome"].isna().sum())

df = df.dropna(subset=["Outcome"]).copy()

df["eligible"] = df["Outcome"]

# CLEAN TARGET VARIABLE

df["eligible"] = pd.to_numeric(
    df["eligible"],
    errors="coerce"
)

df = df.dropna(subset=["eligible"]).copy()

df = df[df["eligible"].isin([0, 1])].copy()

df["eligible"] = df["eligible"].astype(int)

print("\nTarget distribution:")
print(df["eligible"].value_counts())

# FEATURE SELECTION

X = df[FEATURES].copy()

maps = [
    education_map,
    gender_map,
    community_map,
    religion_map,
    yes_no_map,
    yes_no_map,
    yes_no_map,
    percentage_map,
    income_map,
    india_map
]

# ENCODE FEATURES

for col, mapping in zip(FEATURES, maps):

    if X[col].dtype == "object":
        X[col] = X[col].astype(str).str.strip()

    X[col] = X[col].map(mapping)

# CHECK MISSING ENCODED VALUES

print("\nMissing values after encoding:")
print(X.isna().sum())

# REMOVE INVALID FEATURE ROWS

valid_rows = X.notna().all(axis=1)

print("\nInvalid rows removed:", (~valid_rows).sum())

X = X.loc[valid_rows].copy()

y = df.loc[X.index, "eligible"].copy()

X = X.astype(float)

y = y.astype(int)

# FINAL DATA CHECK

print("\nFinal dataset:")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values in X:")
print(X.isna().sum().sum())

print("Missing values in y:")
print(y.isna().sum())

print("\nFinal target distribution:")
print(y.value_counts())

# TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

# MODEL TRAINING

model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

# MODEL PREDICTION

y_pred = model.predict(X_test)

# MODEL EVALUATION

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

print("\nMODEL PERFORMANCE")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

# SHAP EXPLAINER

explainer = shap.TreeExplainer(model)

# SCHOLARSHIP RULE ENGINE

def scholarship_rules(scholarship, data):

    reasons = []

    if data["India"] == 1:
        reasons.append(
            "Scholarship is applicable only to students studying in India."
        )

    is_doctorate = (
        data["Education_Qualification"] == 2
    )

    if scholarship == "State Merit Scholarship":

        if is_doctorate:
            reasons.append(
                "State Merit Scholarship is not available for Doctorate level."
            )

        if data["Annual_Percentage"] < 1:
            reasons.append(
                "Minimum 50% marks required."
            )

        if data["Income"] > 0:
            reasons.append(
                "Family income must be Upto 1.5L."
            )

    elif scholarship == "Minority Scholarship":

        if data["Religion"] == 0:
            reasons.append(
                "Only applicable for Minority religion students."
            )

        if data["Annual_Percentage"] < 1:
            reasons.append(
                "Minimum 50% marks required."
            )

    elif scholarship == "Sports Scholarship":

        if is_doctorate:
            reasons.append(
                "Sports Scholarships are rarely available for Doctorate level."
            )

        if data["Sports"] == 0:
            reasons.append(
                "Active sports participation certificate required."
            )

    elif scholarship == "Disability Scholarship":

        if data["Disability"] == 0:
            reasons.append(
                "Disability certificate required."
            )

    elif scholarship == "Ex-Servicemen Scholarship":

        if is_doctorate:
            reasons.append(
                "Ex-Servicemen scholarships are not available for Doctorate level."
            )

        if data["Exservice_men"] == 0:
            reasons.append(
                "Parental Ex-Servicemen status required."
            )

    return reasons

# ALTERNATIVE SCHOLARSHIP RECOMMENDATION

def recommend_alternatives(
    data,
    selected_scholarship
):

    scholarships = [
        "State Merit Scholarship",
        "Minority Scholarship",
        "Sports Scholarship",
        "Disability Scholarship",
        "Ex-Servicemen Scholarship"
    ]

    alternatives = []

    for scholarship in scholarships:

        if scholarship == selected_scholarship:
            continue

        reasons = scholarship_rules(
            scholarship,
            data
        )

        if not reasons:
            alternatives.append(scholarship)

    return alternatives

# MAIN SCHOLARSHIP SYSTEM

def scholarship_system(
    scholarship,
    education,
    gender,
    community,
    religion,
    exservice,
    disability,
    sports,
    percentage,
    income,
    india_status
):

    data = {

        "Education_Qualification":
            education_map.get(education, 0),

        "Gender":
            gender_map.get(gender, 0),

        "Community":
            community_map.get(community, 0),

        "Religion":
            religion_map.get(religion, 0),

        "Exservice_men":
            yes_no_map.get(exservice, 0),

        "Disability":
            yes_no_map.get(disability, 0),

        "Sports":
            yes_no_map.get(sports, 0),

        "Annual_Percentage":
            percentage_map.get(percentage, 0),

        "Income":
            income_map.get(income, 0),

        "India":
            india_map.get(india_status, 0)
    }

    user_df = pd.DataFrame(
        [data]
    )[FEATURES]

    user_df = user_df.astype(float)

    # ML PREDICTION

    ml_pred = model.predict(user_df)[0]

    # SHAP EXPLANATION

    shap_values = explainer.shap_values(user_df)

    if isinstance(shap_values, list):
        shap_row = shap_values[0]
    else:
        shap_row = shap_values[0]

    feature_importance = dict(
        zip(
            user_df.columns,
            shap_row
        )
    )

    sorted_features = sorted(
        feature_importance.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )

    shap_explanation = "Feature contributions (SHAP):\n"

    for feature, value in sorted_features:
        shap_explanation += (
            f"- {feature}: {value:.3f}\n"
        )

    # RULE VERIFICATION

    rule_reasons = scholarship_rules(
        scholarship,
        data
    )

    # ALTERNATIVE SCHOLARSHIPS

    if rule_reasons or ml_pred == 0:

        alternatives = recommend_alternatives(
            data,
            scholarship
        )

        if alternatives:

            alt_text = (
                "Alternative Scholarships:\n"
                + "\n".join(
                    [f"• {a}" for a in alternatives]
                )
            )

        else:

            alt_text = "No other alternatives found."

    else:

        alt_text = ""

    # FINAL DECISION

    if not rule_reasons and ml_pred == 1:

        status = f"Eligible for {scholarship}"

        explanation = (
            "Human Explanation: All specific "
            "scholarship rules are met.\n\n"
            + shap_explanation
        )

    else:

        status = f"Not Eligible for {scholarship}"

        if rule_reasons:

            explanation = (
                "Failed Requirements:\n"
                + "\n".join(
                    [f"- {r}" for r in rule_reasons]
                )
                + "\n\n"
                + shap_explanation
            )

        else:

            explanation = (
                "Human Explanation: While the "
                "specific scholarship rules are met, "
                "the ML model predicts low eligibility "
                "based on historical trends for this "
                "specific profile.\n\n"
                + shap_explanation
            )

    return status, explanation, alt_text

# GRADIO INTERFACE

interface = gr.Interface(

    fn=scholarship_system,

    inputs=[

        gr.Dropdown(
            [
                "State Merit Scholarship",
                "Minority Scholarship",
                "Sports Scholarship",
                "Disability Scholarship",
                "Ex-Servicemen Scholarship"
            ],
            label="Select Scholarship"
        ),

        gr.Dropdown(
            list(education_map.keys()),
            label="Education"
        ),

        gr.Dropdown(
            list(gender_map.keys()),
            label="Gender"
        ),

        gr.Dropdown(
            list(community_map.keys()),
            label="Community"
        ),

        gr.Dropdown(
            list(religion_map.keys()),
            label="Religion"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Ex-Servicemen Status"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Disability Status"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Sports Participation"
        ),

        gr.Dropdown(
            list(percentage_map.keys()),
            label="Annual Percentage"
        ),

        gr.Dropdown(
            list(income_map.keys()),
            label="Family Income Range"
        ),

        gr.Dropdown(
            ["In", "Out"],
            label="Study Location (Inside India)"
        )
    ],

    outputs=[

        gr.Textbox(
            label="Final Output: Eligibility Status"
        ),

        gr.Textbox(
            label="Human-Friendly Explanation",
            lines=12
        ),

        gr.Textbox(
            label="Personalized Scholarship Suggestions (Alternatives)",
            lines=6
        )
    ],

    title="Explainable Scholarship Eligibility & Recommendation System",

    description=(
        "Based on the research framework for addressing "
        "transparency gaps in state scholarship schemes."
    )
)

interface.launch()

Dataset loaded successfully.
Dataset shape: (200000, 12)

Missing Outcome values before cleaning:
0

Target distribution:
eligible
1    100000
0    100000
Name: count, dtype: int64

Missing values after encoding:
Education_Qualification    57893
Gender                         0
Community                      0
Religion                   49700
Exservice_men                  0
Disability                     0
Sports                         0
Annual_Percentage          59826
Income                         0
India                          0
dtype: int64

Invalid rows removed: 125778

Final dataset:
X shape: (74222, 10)
y shape: (74222,)

Missing values in X:
0
Missing values in y:
0

Final target distribution:
eligible
0    37150
1    37072
Name: count, dtype: int64

Training data: (59377, 10)
Testing data: (14845, 10)

MODEL PERFORMANCE
Accuracy  : 0.7865
Precision : 0.7487
Recall    : 0.8618
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. A